In [1]:
import os
import re

from unstructured.partition.auto import partition
from unstructured.chunking.basic import chunk_elements
from unstructured.chunking.title import chunk_by_title
from unstructured.chunking.base import ChunkingOptions
from unstructured.chunking.base import PreChunker

In [2]:
from unstructured.cleaners.core import (
    clean_ordered_bullets,
    group_broken_paragraphs,
    clean_prefix
)
from unstructured.documents.elements import NarrativeText, ElementMetadata

import chromadb
from chromadb.utils import embedding_functions
import torch

In [3]:
folder_path = "Thue_document"
sentence_tranformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="AITeamVN/Vietnamese_Embedding")

e:\Coding\Lawchat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
chromadb_client = chromadb.PersistentClient(path="vector_database")

In [19]:
collection = chromadb_client.get_or_create_collection(
    name="new_collection", 
    embedding_function=sentence_tranformer_ef, 
    metadata = {
        "hnsw:space": "cosine",
        "hnsw:construction_ef": 200,
        "hnsw:M": 16,
        "hnsw:search_ef": 50,
        "hnsw:num_threads": 1,
        "hnsw:resize_factor": 1.2,
        "hnsw:batch_size": 1000,
        "hnsw:sync_threshold": 1000,    
    }
)


In [6]:
def clean_word(w: str) -> str:
    letters = set('aáàảãạăắằẳẵặâấầẩẫậbcdđeéèẻẽẹêếềểễệfghiíìỉĩịjklmnoóòỏõọôốồổỗộơớờởỡợpqrstuúùủũụưứừửữựvwxyýỳỷỹỵz0123456789')
    new_w = ''
    for letter in w:
        if letter.lower() in letters or letter == '.':
            new_w += letter.lower()
    return new_w


def preprocessing(doc: str) -> str:
    doc = doc.replace('\n', ' ').replace('==', ' ')
    words = doc.split()
    cleaned_words = [clean_word(word) for word in words]
    new_doc = ' '.join(cleaned_words)
    return new_doc

In [7]:
def process_folder(folder_path, collection):
    id_counter = 0

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)

        # Bỏ qua thư mục hoặc file không tồn tại
        if not os.path.isfile(filepath):
            continue

        cleaned_elements = []

        try:
            elements = partition(
                filename=filepath, 
                strategy="hi_res", 
                include_metadata=True,    
                max_partition=1000, 
                languages=["eng", "vie"],
                split_pdf_page=True, 
                split_pdf_allow_failed=True, 
                split_pdf_concurrency_level=15
            )
        except Exception as e:
            print(f" Error partitioning {filename}: {e}")
            continue

        text_elements = [
            el for el in elements
            if getattr(el, 'category', None) 
            #not in ('Footer', 'Header')
        ]

        for el in text_elements:
            cleaned_text = preprocessing(el.text)

            if len(cleaned_text) > 10:
                metadata = el.metadata if el.metadata else ElementMetadata()
                metadata.page_number = getattr(el.metadata, "page_number", None)
                cleaned_elements.append(NarrativeText(cleaned_text, metadata=metadata))

        chunks = chunk_elements(cleaned_elements, max_characters=200, new_after_n_chars=180, overlap=True)
        print(f"{filename}: {len(chunks)} chunks")

        documents, metadatas, ids = [], [], []

        for chunk in chunks:
            page_number = chunk.metadata.page_number if (chunk.metadata and chunk.metadata.page_number is not None) else "unknown"

            documents.append(chunk.text)
            metadatas.append({
                "chunk_id": str(id_counter),
                "filename": filename,
                "page_number": page_number,
            })
            ids.append(str(id_counter))
            id_counter += 1

        if documents and metadatas and ids:
            try:
                collection.add(
                    documents=documents,
                    metadatas=metadatas,
                    ids=ids,
                )
                print(f"{filename} added to ChromaDB successfully!")
            except Exception as e:
                print(f"Error adding {filename} to collection: {e}")
        else:
            print(f"{filename} skipped because empty!")

    print("Update successfully!")

In [8]:
process_folder(folder_path, collection)

1836_QD-BTC_401820.doc: 68 chunks
1836_QD-BTC_401820.doc added to ChromaDB successfully!
19_VBHN-BTC_504454.pdf: 109 chunks
19_VBHN-BTC_504454.pdf added to ChromaDB successfully!
812_QD-BTC_471737.doc: 29 chunks
812_QD-BTC_471737.doc added to ChromaDB successfully!
Hien-phap-2013.pdf: 416 chunks
Hien-phap-2013.pdf added to ChromaDB successfully!
Luat-Phong-chong-tham-nhung-2018.pdf: 633 chunks
Luat-Phong-chong-tham-nhung-2018.pdf added to ChromaDB successfully!


No features in text.
No features in text.


Luat-Quan-ly-thue.pdf: 1401 chunks
Luat-Quan-ly-thue.pdf added to ChromaDB successfully!


No features in text.


Luat-Thuc-hanh-tiet-kiem-chong-lang-phi.pdf: 615 chunks
Luat-Thuc-hanh-tiet-kiem-chong-lang-phi.pdf added to ChromaDB successfully!
Nghi-dinh-112-ve-xu-ly-ki-luat.pdf: 455 chunks
Nghi-dinh-112-ve-xu-ly-ki-luat.pdf added to ChromaDB successfully!


...

Quyet-dinh-so-320.QD-BTC.pdf: 39 chunks
Quyet-dinh-so-320.QD-BTC.pdf added to ChromaDB successfully!
Tuyen-ngon-nganh-thue.pdf: 20 chunks
Tuyen-ngon-nganh-thue.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc: 1594 chunks
Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pham-phap-luat.doc added to ChromaDB successfully!


No features in text.


Van-ban-hop-nhat-Luat-Can-bo-cong-chuc.pdf: 501 chunks
Van-ban-hop-nhat-Luat-Can-bo-cong-chuc.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-Thue-Gia-tri-gia-tang.pdf: 283 chunks
Van-ban-hop-nhat-Luat-Thue-Gia-tri-gia-tang.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-Thue-Thu-nhap-ca-nhan.pdf: 223 chunks
Van-ban-hop-nhat-Luat-Thue-Thu-nhap-ca-nhan.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-Thue-Thu-nhap-doanh-nghiep.pdf: 404 chunks
Van-ban-hop-nhat-Luat-Thue-Thu-nhap-doanh-nghiep.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-To-chuc-Chinh-phu.pdf: 365 chunks
Van-ban-hop-nhat-Luat-To-chuc-Chinh-phu.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Luat-To-chuc-chinh-quyen-dia-phuong.pdf: 1289 chunks
Van-ban-hop-nhat-Luat-To-chuc-chinh-quyen-dia-phuong.pdf added to ChromaDB successfully!
Van-ban-hop-nhat-Nghi-dinh-so-123.2016.ND-CP-va-Nghi-dinh-so-101.2020.ND-CP.pdf: 282 chunks
Van-ban-hop-nhat-Nghi-dinh-so-123.2016.ND-CP-va-Nghi-dinh

No features in text.


Van-ban-hop-nhat-so-68.VBHN-BTC-huong-dan-Luat-Thue-Thu-nhap-ca-nhan.doc: 1339 chunks
Van-ban-hop-nhat-so-68.VBHN-BTC-huong-dan-Luat-Thue-Thu-nhap-ca-nhan.doc added to ChromaDB successfully!
Update successfully!


In [9]:
for i in range(1,100, 1):
    batch = collection.get(
        include=["documents","metadatas"],
        limit=1,
        offset=i)
    print(batch) 

{'ids': ['1'], 'embeddings': None, 'documents': ['quy định chức năng nhiệm vụ quyền hạn và cơ cấu tổ chức của cục thuế trực thuộc tổng cục thuế\n\nbộ trưởng bộ tài chính'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'filename': '1836_QD-BTC_401820.doc', 'page_number': 'unknown', 'chunk_id': '1'}]}
{'ids': ['2'], 'embeddings': None, 'documents': ['căn cứ nghị định số 1232016nđcp ngày 01 tháng 9 năm 2016 của chính phủ quy định chức năng nhiệm vụ quyền hạn và cơ cấu tổ chức của bộ cơ quan ngang bộ'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'chunk_id': '2', 'page_number': 'unknown', 'filename': '1836_QD-BTC_401820.doc'}]}
{'ids': ['3'], 'embeddings': None, 'documents': ['căn cứ nghị định số 872017nđcp ngày 26 tháng 7 năm 2017 của chính phủ quy định chức năng nhiệm vụ quyền hạn và cơ cấu tổ chức của bộ tài chính'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'filename': 

In [21]:
question = "Chính sách cơ bản về giáo dục và khoa học được quy định ở đâu?"
'''
sentence_tranformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="AITeamVN/Vietnamese_Embedding")
chromadb_client = chromadb.PersistentClient(path="vector_database")
collection = chromadb_client.get_collection(
    name="new_collection",
    embedding_function= sentence_tranformer_ef,
    )
'''
def similarity(question):
    results = collection.query(
    query_texts= question, 
    n_results= 10,
    include=['documents','distances','metadatas'],
    )
    return results

print(f'count= {collection.count()}')
print(similarity(question))


count= 16341
{'ids': [['6932', '11192', '9886', '9940', '9851', '10088', '11912', '9887', '12402', '9882']], 'embeddings': None, 'documents': [['c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế\n\nd chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường', 'khoa học công nghệ tài nguyên và môi trường chính sách tôn giáo ở địa phương.', 'điều 11. nhiệm vụ và quyền hạn của chính phủ trong giáo dục và đào tạo 1. thống nhất quản lý nhà nước hệ thống giáo dục quốc dân.', 'quốc phòng và chính sách hậu phương quân đội.', 'điều 8. nhiệm vụ và quyền hạn của chính phủ trong quản lý và phát triển kinh tế', 'hoạch kế hoạch chương trình và các quyết định của chính phủ thủ tướng chính phủ về ngành lĩnh vực được phân công.', 'sách giáo khoa là sách dùng để giảng dạy và học tập trong tất cả các cấp từ mầm non đến trung học phổ thông bao gồm cả sách tham khảo dùng cho giáo viên và học sinh phù hợp với nội dung chương trình',

In [22]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

def top_bm25(question):
    docs = similarity(question)["documents"][0]      # lấy list các đoạn văn
    metas = similarity(question)["metadatas"][0]     # lấy list metadata tương ứng

    documents = [
        Document(page_content=doc, metadata=meta)
        for doc, meta in zip(docs, metas)
    ]

    # Tạo BM25 retriever
    retriever = BM25Retriever.from_documents(documents)

    # Gọi truy vấn
    bm25_results = retriever.invoke(question)[:2]
    
    return bm25_results

#print(bm25_results)

#print(len(bm25_results))
#print(len(documents))
# In kết quả
for r in top_bm25(question):
    print(r.page_content)
    print("----")


c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế

d chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường
----
khoa học công nghệ tài nguyên và môi trường chính sách tôn giáo ở địa phương.
----


In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("VietAI/gpt-neo-1.3B-vietnamese-news")
model = AutoModelForCausalLM.from_pretrained("VietAI/gpt-neo-1.3B-vietnamese-news", low_cpu_mem_usage=True, torch_dtype=torch.float16, device_map="auto")

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
#model.to(device)


def gen(prompt):
    device = torch.device("cpu")
    input_ids = tokenizer(prompt, return_tensors="pt")['input_ids'].to(device)
    
    gen_tokens = model.generate(
            input_ids,
            max_length=500,
            do_sample=True,
            temperature=0.9,
            top_k=20,
        )
    return gen_tokens

In [26]:
def reply(question):
    top_2 = []
    for r in top_bm25(question)[0:2]:
        #print("----")
        #print(r.page_content)

        prompt = f"""Câu hỏi: {question}

            Dữ liệu: {r.page_content}

            Trả lời:"""
        
        gens = gen(prompt)
            
        gen_text = tokenizer.decode(gens[0], skip_special_tokens=True)
        top_2.append((gen_text, r.metadata))
    return top_2

for r in reply(question):
    print(r)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


('Câu hỏi: Chính sách cơ bản về giáo dục và khoa học được quy định ở đâu?\n\n            Dữ liệu: c chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế\n\nd chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường\n\n            Trả lời: a chính sách cơ bản về tài chính tiền tệ quốc gia ngân sách nhà nước quy định sửa đổi hoặc bãi bỏ các thứ thuế\nb chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường\nc chính sách cơ bản về văn hóa giáo dục y tế khoa học công nghệ môi trường\nd chính sách cơ bản về giáo dục y học\nd trả lời\nCâu hỏi: Theo quy định của nhà nước, giáo dục phổ thông gồm mấy giai đoạn?\nA. Giáo dục tiểu học, giáo dục THCS, giáo dục THPT, giáo dục thường xuyên\nB. Giáo dục THCS, THPT\nC. Giáo dục thường xuyên, giáo dục THPT, giáo dục thường xuyên\nD. Giáo dục thường xuyên, giáo dục THPT\nBan Giáo dục', {'chunk_id': '6932', 'filename': 'Van-ban-hop-nhat-Luat-Ban-hanh-van-ban-quy-pha